# Importing Libraries

In [1]:
import random
import pandas as pd
import json
import re
import streamlit as st
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
import joblib
import requests
import time
from openai import OpenAI
from attackcti import attack_client
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import uuid
import datetime

# Creating SYNTHETIC Realistic Multi-Source Schema

# We simulate data coming from:

# network logs
# auth logs
# application logs

In [2]:
def generate_ip():
    return f"{random.randint(1,255)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(0,255)}"

def generate_sample_type(attack_type):

    if attack_type == "Brute Force":
        return {
            "ip": generate_ip(),
            "failed_logins": random.randint(8, 30) if random.random() > 0.2 else random.randint(0, 5),
            "successful_logins": random.randint(0, 2),
            "bytes_sent": random.randint(1000, 5000),
            "bytes_received": random.randint(1000, 5000),
            "request_rate": random.randint(10, 30),
            "unique_ports": random.randint(1, 5),
            "is_port_scan": 0,
            "suspicious_process": 0,
            "dns_queries": random.randint(10, 50),
            "label": "Brute Force"
        }

    elif attack_type == "DoS":
        return {
            "ip": generate_ip(),
            "failed_logins": random.randint(0, 2),
            "successful_logins": random.randint(0, 2),
            "bytes_sent": random.randint(50000, 200000),
            "bytes_received": random.randint(50000, 200000),
            "request_rate": random.randint(100, 500),
            "unique_ports": random.randint(1, 5),
            "is_port_scan": 0,
            "suspicious_process": 0,
            "dns_queries": random.randint(50, 200),
            "label": "DoS"
        }

    elif attack_type == "Port Scan":
        return {
            "ip": generate_ip(),
            "failed_logins": 0,
            "successful_logins": 0,
            "bytes_sent": random.randint(1000, 5000),
            "bytes_received": random.randint(1000, 5000),
            "request_rate": random.randint(20, 80),
            "unique_ports": random.randint(20, 100),
            "is_port_scan": 1,
            "suspicious_process": 0,
            "dns_queries": random.randint(10, 50),
            "label": "Port Scan"
        }

    else:  # Normal
        return {
            "ip": generate_ip(),
            "failed_logins": random.randint(0, 2),
            "successful_logins": random.randint(1, 10),
            "bytes_sent": random.randint(1000, 10000),
            "bytes_received": random.randint(1000, 10000),
            "request_rate": random.randint(1, 20),
            "unique_ports": random.randint(1, 10),
            "is_port_scan": 0,
            "suspicious_process": 0,
            "dns_queries": random.randint(5, 30),
            "label": "Normal"
        }

# Generating dataset

In [3]:
data = []
for _ in range(2500):
    data.append(generate_sample_type("Normal"))
    data.append(generate_sample_type("Brute Force"))
    data.append(generate_sample_type("DoS"))
    data.append(generate_sample_type("Port Scan"))

df = pd.DataFrame(data)

In [4]:
df.head()

,ip,failed_logins,successful_logins,bytes_sent,bytes_received,request_rate,unique_ports,is_port_scan,suspicious_process,dns_queries,label
0,243.26.80.217,0,8,4785,5773,12,5,0,0,6,Normal
1,131.192.114.52,3,2,1202,1327,24,5,0,0,28,Brute Force
2,91.140.162.29,2,2,131619,147069,334,2,0,0,84,DoS
3,230.126.74.7,0,0,4074,3510,75,43,1,0,12,Port Scan
4,253.27.227.113,2,6,7856,7809,15,5,0,0,29,Normal


# TRAINING ML MODEL

In [4]:
X = df.drop(columns=["label", "ip"])
y = df["label"]

feature_columns = X.columns  # IMPORTANT

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier()
model.fit(X_train, y_train)

print("Accuracy:", model.score(X_test, y_test))

Accuracy: 0.997


# TRAINING ANOMALY MODEL

In [5]:
normal_df = df[df["label"] == "Normal"]
X_normal = normal_df.drop(columns=["label", "ip"])

iso_model = IsolationForest(n_estimators=100, contamination=0.1, random_state=42)
iso_model.fit(X_normal)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.1
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [6]:
joblib.dump(model, "C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/rf_model.pkl")
joblib.dump(iso_model, "C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/iso_model.pkl")
joblib.dump(feature_columns, "C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/features.pkl")

print("Models saved successfully!")

Models saved successfully!


In [2]:
model = joblib.load("C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/rf_model.pkl")
iso_model = joblib.load("C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/iso_model.pkl")
feature_columns = joblib.load("C:/Users/ashee/Cyber AI Projects/P1_AI_SOC_ANALYST/features.pkl")

# FEATURE PARSER

In [3]:
def log_to_features(log):
    log_lower = log.lower()

    ip_match = re.search(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b', log)
    ip = ip_match.group(0) if ip_match else "0.0.0.0"

    numbers = re.findall(r'\d+', log)
    count = int(numbers[-1]) if numbers else 0

    features = {
        "failed_logins": 0,
        "successful_logins": 0,
        "bytes_sent": 5000,
        "bytes_received": 5000,
        "request_rate": 10,
        "unique_ports": 1,
        "is_port_scan": 0,
        "suspicious_process": 0,
        "dns_queries": 10
    }

    # 🔐 Login patterns
    if ("fail" in log_lower and "login" in log_lower):
        features["failed_logins"] = count

        if count <= 3:
            features["request_rate"] = random.randint(1, 5)
        elif count <= 7:
            features["request_rate"] = random.randint(5, 15)
        else:
            features["request_rate"] = random.randint(20, 50)

    # 🌐 Port scan
    if "port" in log_lower and "scan" in log_lower:
        features["is_port_scan"] = 1
        features["unique_ports"] = count if count > 0 else 50

    # 💥 DoS detection (STRONGER)
    if ("dos" in log_lower or 
        "flood" in log_lower or 
        "traffic spike" in log_lower or 
        "high traffic" in log_lower):

        features["request_rate"] = max(count, 300)
        features["bytes_sent"] = 200000
        features["dns_queries"] = 300

    if "per second" in log_lower or "rapid" in log_lower:
        features["request_rate"] = count * 2

    # Suspicious IP simulation
    if ip.startswith("45"):
        features["dns_queries"] += 20

    return ip, features

# RULE ENGINE

In [4]:
def apply_rules(features):

    # Brute force
    if features["failed_logins"] > 7:
        return "Brute Force (Rule)"

    # Port scan
    if features["is_port_scan"] == 1:
        return "Port Scan (Rule)"

    # 🚨 NEW: DoS RULE
    if features["request_rate"] > 200:
        return "DoS (Rule)"

    return "No Rule Triggered"

# ANOMALY SCORE

In [5]:
def get_anomaly_score(model, df):
    raw_score = model.decision_function(df)[0]

    # better normalization
    normalized = max(0, min(1, (0.7 - raw_score)))

    return round(float(normalized), 2)

# FINAL DECISION LOGIC

In [6]:
def final_decision(prediction, confidence, rule, anomaly_score, features):

    # 🚫 prevent false positives
    if features["failed_logins"] <= 3 and confidence > 0.8:
        return "Suspicious", "Medium"

    # 🚨 DoS rule priority
    if "DoS" in rule:
        return "DoS", "High"

    # 🚨 anomaly
    if anomaly_score > 0.6:
        return "Anomalous Activity", "High"

    # 🚨 brute force
    if "Brute Force" in rule:
        return "Brute Force", "High"

    # 🎯 ML
    if confidence > 0.85:
        return prediction, "High"

    if confidence > 0.7:
        return prediction, "Medium"

    if confidence > 0.5:
        return "Suspicious", "Medium"

    return "Suspicious", "Low"


# MAIN PIPELINE

In [7]:
def analyze_log(log):
    ip, features = log_to_features(log)

    df_input = pd.DataFrame([features])
    df_input = df_input[feature_columns]

    prediction = model.predict(df_input)[0]
    confidence = model.predict_proba(df_input).max()

    rule = apply_rules(features)
    anomaly_score = get_anomaly_score(iso_model, df_input)

    final_label, severity = final_decision(
        prediction, confidence, rule, anomaly_score, features
    )

    return {
        "IP Address": ip,
        "Prediction": prediction,
        "Confidence": round(float(confidence), 2),
        "Rule Triggered": rule,
        "Anomaly Score": anomaly_score,
        "Final Decision": final_label,
        "Severity": severity
    }

# TEST

In [8]:
tests = [
    "Failed login attempts from IP 45.12.23.11 20 times",
    "20 failed logins from 45.12.23.11",
    "Login failures (20) detected for IP 45.12.23.11",
    "User failed to login 20 times from 45.12.23.11"
]

for t in tests:
    print(analyze_log(t))

{'IP Address': '45.12.23.11', 'Prediction': 'Brute Force', 'Confidence': 0.98, 'Rule Triggered': 'Brute Force (Rule)', 'Anomaly Score': 0.74, 'Final Decision': 'Anomalous Activity', 'Severity': 'High'}
{'IP Address': '45.12.23.11', 'Prediction': 'Brute Force', 'Confidence': 0.92, 'Rule Triggered': 'Brute Force (Rule)', 'Anomaly Score': 0.74, 'Final Decision': 'Anomalous Activity', 'Severity': 'High'}
{'IP Address': '45.12.23.11', 'Prediction': 'Brute Force', 'Confidence': 0.92, 'Rule Triggered': 'Brute Force (Rule)', 'Anomaly Score': 0.74, 'Final Decision': 'Anomalous Activity', 'Severity': 'High'}
{'IP Address': '45.12.23.11', 'Prediction': 'Brute Force', 'Confidence': 0.98, 'Rule Triggered': 'Brute Force (Rule)', 'Anomaly Score': 0.74, 'Final Decision': 'Anomalous Activity', 'Severity': 'High'}


In [9]:
#print(analyze_log("Failed login attempts from IP 45.12.23.11 3 times"))
print(analyze_log("Failed login attempts from IP 185.220.101.1 1 times"))

{'IP Address': '185.220.101.1', 'Prediction': 'Normal', 'Confidence': 0.57, 'Rule Triggered': 'No Rule Triggered', 'Anomaly Score': 0.68, 'Final Decision': 'Anomalous Activity', 'Severity': 'High'}


In [9]:
layer2_output = analyze_log("Failed login attempts from IP 185.220.101.1 10 times")

# Setting API Keys

In [10]:
# ABUSEIPDB_API_KEY = "a5657------------------------------------------5f6478e623"
# OTX_API_KEY = "5ea010-----------------------------------------------f322d692d9c82d0"

# AbuseIPDB Function

In [11]:
def get_abuseipdb_data(ip):
    url = "https://api.abuseipdb.com/api/v2/check"
    
    headers = {
        "Key": ABUSEIPDB_API_KEY,
        "Accept": "application/json"
    }
    
    params = {
        "ipAddress": ip,
        "maxAgeInDays": 90
    }

    try:
        response = requests.get(url, headers=headers, params=params, timeout=5)
        data = response.json()

        if "data" in data:
            return {
                "abuse_score": data["data"]["abuseConfidenceScore"],
                "reports": data["data"]["totalReports"],
                "country": data["data"]["countryCode"]
            }
    except Exception as e:
        print("AbuseIPDB error:", e)

    return None

# OTX Function

In [12]:
def get_otx_data(ip):
    url = f"https://otx.alienvault.com/api/v1/indicators/IPv4/{ip}/general"
    
    headers = {
        "X-OTX-API-KEY": OTX_API_KEY
    }

    try:
        response = requests.get(url, headers=headers, timeout=5)
        data = response.json()

        pulses = data.get("pulse_info", {}).get("pulses", [])
        
        return {
            "pulse_count": len(pulses),
            "malware_families": list(set(
                tag for pulse in pulses for tag in pulse.get("tags", [])
            ))[:5]  # limit for readability
        }
    except Exception as e:
        print("OTX error:", e)

    return None

# Scoring Logic

In [13]:
def calculate_reputation(abuse_data, otx_data):
    score = 0

    # AbuseIPDB weight (strong signal)
    if abuse_data:
        score += abuse_data["abuse_score"] * 0.6

    # OTX weight (context signal)
    if otx_data:
        score += min(otx_data["pulse_count"] * 4, 40)

    score = min(int(score), 100)

    # Classification
    if score >= 75:
        label = "Malicious"
    elif score >= 40:
        label = "Suspicious"
    else:
        label = "Clean"

    return label, score

# Main Function

In [14]:
def fuse_decision(layer2_output, reputation_label):
    final_decision = layer2_output["Final Decision"]
    severity = layer2_output["Severity"]

    # Case 1: ML says bad BUT intel says clean
    if final_decision == "Anomalous Activity" and reputation_label == "Clean":
        severity = "Medium"   # downgrade
        note = "No external threat intel found"

    # Case 2: Both agree it's bad
    elif final_decision == "Anomalous Activity" and reputation_label == "Malicious":
        severity = "Critical"
        note = "Confirmed by threat intelligence"

    # Case 3: Intel says bad but ML missed
    elif final_decision == "Normal" and reputation_label == "Malicious":
        final_decision = "Suspicious Activity"
        severity = "High"
        note = "Flagged by external threat intelligence"

    else:
        note = "No significant correlation"

    return final_decision, severity, note

In [15]:
def layer3_threat_intel(layer2_output):
    ip = layer2_output.get("IP Address")

    if not ip:
        return {"error": "No IP Address found in input"}

    print(f"🔍 Enriching IP: {ip}")

    abuse_data = get_abuseipdb_data(ip)
    time.sleep(1)  # avoid rate limit

    otx_data = get_otx_data(ip)

    label, score = calculate_reputation(abuse_data, otx_data)
    new_decision, new_severity, note = fuse_decision(layer2_output, label)


    enrichment = {
        "Threat Intel": {
            "AbuseIPDB": abuse_data,
            "OTX": otx_data,
            "IP Reputation": f"{label} ({score}%)",
            "Fusion Note": note
        },
        "Final Decision": new_decision,
        "Severity": new_severity
    }

    return enrichment

# FINAL OUTPUT

In [16]:
layer3_output = layer3_threat_intel(layer2_output)

# Merge results
final_output = {**layer2_output, **layer3_output}

final_output

🔍 Enriching IP: 185.220.101.1
OTX error: HTTPSConnectionPool(host='otx.alienvault.com', port=443): Read timed out. (read timeout=5)


{'IP Address': '185.220.101.1',
 'Prediction': 'Brute Force',
 'Confidence': 0.91,
 'Rule Triggered': 'Brute Force (Rule)',
 'Anomaly Score': 0.7,
 'Final Decision': 'Anomalous Activity',
 'Severity': 'High',
 'Threat Intel': {'AbuseIPDB': {'abuse_score': 100,
   'reports': 139,
   'country': 'DE'},
  'OTX': None,
  'IP Reputation': 'Suspicious (60%)',
  'Fusion Note': 'No significant correlation'}}

# For Clean LLM Rsponse in JSON

In [17]:
def clean_llm_json(response_text):
    # Remove ```json ``` wrappers
    cleaned = re.sub(r"```json|```", "", response_text).strip()
    
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        print("⚠️ JSON parsing failed. Raw output returned.")
        return {"raw_output": response_text}

# Loading MITRE ATT&CK

In [18]:
lift = attack_client()

techniques = lift.get_techniques()

mitre_data = []

for t in techniques:
    try:
        name = t.get("name", "")
        description = t.get("description", "")
        
        for ref in t.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                mitre_id = ref.get("external_id")
                
                mitre_data.append({
                    "name": name,
                    "description": description,
                    "mitre_id": mitre_id
                })
    except:
        continue

# Building RAG Index (FAISS)

In [19]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

documents = [item["description"] for item in mitre_data]
embeddings = embedder.encode(documents)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

def retrieve_mitre_context(query, top_k=1):
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec), top_k)
    
    results = [mitre_data[i] for i in indices[0]]
    return results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# Main Function

In [20]:
# client = OpenAI(api_key="sk-proj-wjYSA0-----------------------------------------XEq9UvCd-ZLAblBDmDNMA")

def layer4_llm_rag(layer3_output):

    # Extracting info
    attack_type = layer3_output.get("Prediction")
    ip = layer3_output.get("IP Address")
    severity = layer3_output.get("Severity")
    anomaly_score = layer3_output.get("Anomaly Score")
    threat_intel = layer3_output.get("Threat Intel")

    # -------------------------------
    #  RAG Retrieval (MITRE)
    # -------------------------------
    query = f"{attack_type} attack behavior"
    mitre_context = retrieve_mitre_context(query, top_k=1)
    mitre_name = mitre_context[0]["name"]
    mitre_id = mitre_context[0]["mitre_id"]
    mitre_desc = mitre_context[0]["description"]
    mitre_full = f"{mitre_id} – {mitre_name}"

    # -------------------------------
    #   LLM Prompt
    # -------------------------------
    prompt = f"""
You are a SOC security analyst.

Analyze this security event:

IP Address: {ip}
Attack Type: {attack_type}
Severity: {severity}
Anomaly Score: {anomaly_score}

Threat Intelligence:
{threat_intel}

MITRE ATT&CK Context:
Technique ID: {mitre_id}
Description: {mitre_desc}

Tasks:
1. Explain the incident clearly
2. Validate or adjust severity (Low/Medium/High)
3. Recommend actions
4. Confirm MITRE mapping

Return STRICT JSON format:
{{
  "Incident Explanation": "...",
  "Severity Assessment": "...",
  "Recommended Actions": ["...", "..."],
  "MITRE Technique": "..."
}}
"""

    # -------------------------------
    # Calling OpenAI LLM
    # -------------------------------
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    llm_output = response.choices[0].message.content
    parsed_output = clean_llm_json(llm_output)
    return {
        **parsed_output,
        "MITRE Technique": mitre_full,
        "MITRE ID": mitre_id,
        "MITRE Description": mitre_desc
    }


# Running Layer 4

In [21]:
layer4_output = layer4_llm_rag(final_output)

print(layer4_output)

{'Incident Explanation': 'A brute force attack has been detected originating from the IP address 185.220.101.1, which has a high abuse score of 100 and has been reported 139 times on AbuseIPDB. The attack is characterized by systematic attempts to guess passwords for accounts, potentially indicating an attempt to gain unauthorized access. The IP address is located in Germany and has a suspicious reputation score of 60%. The anomaly score of 0.7 indicates a significant deviation from normal behavior, further supporting the likelihood of malicious intent.', 'Severity Assessment': 'High', 'Recommended Actions': ['Block the IP address 185.220.101.1 at the firewall to prevent further access attempts.', 'Monitor logs for any successful logins or unusual account activity related to this attack.', 'Implement account lockout policies to mitigate the risk of brute force attacks.', 'Notify affected users to change their passwords and enable multi-factor authentication (MFA) where possible.', 'Con

# -----------------------------
# Simulated Integrations
# -----------------------------


In [22]:
def firewall_block_ip(ip):
    return f"[Firewall] Blocked IP: {ip}"

def send_alert(message, severity):
    return f"[Alerting] Alert sent | Severity: {severity} | Message: {message}"

def update_ticket(status, details):
    return f"[Ticketing] Status updated to '{status}' | Details: {details}"

# Playbook Definitions
# -----------------------------

In [23]:
SOAR_PLAYBOOKS = {
    "High": [
        {"action": "block_ip"},
        {"action": "send_alert"},
        {"action": "update_ticket", "status": "Escalated"}
    ],
    "Medium": [
        {"action": "send_alert"},
        {"action": "update_ticket", "status": "Investigating"}
    ],
    "Low": [
        {"action": "update_ticket", "status": "Monitoring"}
    ]
}

# SOAR Engine
# -----------------------------


In [29]:
def execute_playbook(layer3_output, layer4_output):
    incident_id = str(uuid.uuid4())
    response_log = []
    actions_performed = []

    ip = layer3_output.get("IP Address")
    severity = layer3_output.get("Severity", "Low")
    incident_desc = layer4_output.get("Incident Explanation")
    attack_type = layer3_output.get("Prediction", "Unknown")

    confidence = layer3_output.get("Confidence", 0)
    anomaly = layer3_output.get("Anomaly Score", 0)
    abuse = layer3_output.get("Threat Intel", {}).get("AbuseIPDB", {}).get("abuse_score", 0)
    risk_score = round((confidence * 0.4 + anomaly * 0.3 + (abuse / 100) * 0.3), 2)
    
    playbook = SOAR_PLAYBOOKS.get(severity, SOAR_PLAYBOOKS["Low"])

    for step in playbook:
        action = step["action"]

        if action == "block_ip":
            result = firewall_block_ip(ip)
            actions_performed.append("Block IP")
        
        elif action == "send_alert":
            result = send_alert(incident_desc, severity)
            actions_performed.append("Send Alert")
        
        elif action == "update_ticket":
            status = step.get("status", "Open")
            result = update_ticket(status, incident_desc)
            actions_performed.append(f"Update Ticket ({status})")
        
        else:
            result = f"[Unknown Action] {action}"

        failed = any(log["status"] != "success" for log in response_log)
        if failed:
            incident_status = "Failed"

        elif severity == "High":
            incident_status = "Escalated"

        elif severity == "Medium":
            incident_status = "Investigating"

        else:
            incident_status = "Resolved"
        log_entry = {
            "timestamp": datetime.datetime.now().isoformat(),
            "action": action,
            "executor": "SOAR Engine v1.0",
            "status": "success",
            "result": result
        }
        time.sleep(0.2)

        response_log.append(log_entry)

    return {
        "Incident ID": incident_id,
        "Playbook Executed": f"{attack_type} - {severity} Severity Playbook",
        "Risk Score": risk_score,
        "MITRE Technique": layer4_output.get("MITRE Technique"),
        "Actions Performed": actions_performed,
        "Incident Status": playbook[-1].get("status", "Completed"),
        "Response Log": response_log
    }


# Running SOAR

In [30]:
soar_output = execute_playbook(final_output, layer4_output)

soar_output

{'Incident ID': '5ae6f541-869e-4ad3-ae71-f8706692e85c',
 'Playbook Executed': 'Brute Force - High Severity Playbook',
 'Risk Score': 0.87,
 'MITRE Technique': 'T1110 – Brute Force',
 'Actions Performed': ['Block IP', 'Send Alert', 'Update Ticket (Escalated)'],
 'Incident Status': 'Escalated',
 'Response Log': [{'timestamp': '2026-04-26T18:19:51.730041',
   'action': 'block_ip',
   'executor': 'SOAR Engine v1.0',
   'status': 'success',
   'result': '[Firewall] Blocked IP: 185.220.101.1'},
  {'timestamp': '2026-04-26T18:19:51.930990',
   'action': 'send_alert',
   'executor': 'SOAR Engine v1.0',
   'status': 'success',
   'result': '[Alerting] Alert sent | Severity: High | Message: A brute force attack has been detected originating from the IP address 185.220.101.1, which has a high abuse score of 100 and has been reported 139 times on AbuseIPDB. The attack is characterized by systematic attempts to guess passwords for accounts, potentially indicating an attempt to gain unauthorized acc